In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from lanchain.agents.agent import AgentExecutor

ImportError: cannot import name 'create_tool_calling_agent' from 'langchain.agents' (C:\Users\arnav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\langchain\agents\__init__.py)

In [ ]:
df = pd.read_csv('credit_risk_dataset.csv')
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df["loan_int_rate"] = df.groupby("loan_grade")["loan_int_rate"].transform(lambda x: x.fillna(x.mean()))
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())

In [ ]:
df.isnull().sum()

In [ ]:
df.drop(df[df['person_emp_length'] >= 60].index, inplace=True)
df.drop(df[df['person_age'] > 100].index, inplace=True)

In [ ]:
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'Y': 1, 'N': 0})

df['loan_grade'] = df['loan_grade'].map({'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7})

df = pd.get_dummies(df, columns = ['person_home_ownership', 'loan_intent'], drop_first =True)

In [ ]:
df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('loan_status', axis=1), df['loan_status'], test_size=0.2, random_state=42, stratify=df['loan_status'])

In [ ]:
model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
model.fit(X_train, y_train)

In [ ]:
model_predictions = model.predict(X_test)
model_predictions

In [ ]:
acc = accuracy_score(y_test, model_predictions)
acc

In [ ]:
cm = confusion_matrix(y_test, model_predictions)
cm_df = pd.DataFrame(cm, index=['Actual Negative', 'Actual Positive'], columns=['Predicted Negative', 'Predicted Positive'])
cm_df

In [ ]:
ra = roc_auc_score(y_test, model_predictions)
ra

In [ ]:
#%pip install shap
import pandas as pd
import shap

explainer = shap.TreeExplainer(model)

@tool
def credit_risk_assesment(applicant_data: dict) -> dict:
    applicant_df = pd.DataFrame([applicant_data])
    applicant_df = pd.get_dummies(applicant_df, columns=['person_home_ownership', 'loan_intent'], drop_first=True)
    applicant_df = applicant_df.reindex(columns=X_train.columns, fill_value=0)
    
    default_prob = model.predict_proba(applicant_df)[0, 1].item()
    
    shap_vals = explainer(applicant_df)
    
    shap_df = pd.DataFrame({
        'feature': applicant_df.columns,
        'impact': shap_vals.values[0]
    }).sort_values(by='impact', key=abs, ascending=False)
    
    top_drivers = []
    for _, row in shap_df.head(3).iterrows():
        direction = "increased default risk" if row['impact'] > 0 else "reduced default risk"
        top_drivers.append(f"{row['feature']} ({direction} by {abs(row['impact']):.3f})")
    
    return {
        "default_probability_percent": round(default_prob * 100, 2),
        "model_risk_level": "HIGH" if default_prob >= 0.35 else "LOW",
        "top_shap_drivers": top_drivers
    }

In [ ]:
def credit_risk_assessment_tool(applicant_data: dict) -> dict:
    return credit_risk_assesment(applicant_data)

In [ ]:
test_data = {
    'person_age': 25,
    'person_income': 55000,
    'person_emp_length': 3,
    'loan_grade': 2,
    'loan_amnt': 15000,
    'loan_int_rate': 11.5,
    'loan_percent_income': 0.27,
    'cb_person_default_on_file': 0,
    'cb_person_cred_hist_length': 4,
    'person_home_ownership': 'RENT',
    'loan_intent': 'PERSONAL'
}

credit_risk_assessment_tool(test_data)

In [ ]:
@tool
def check_hard_policy_rules(loan_perc_income: float, age: int, loan_amt: float) -> dict:
    violations = []

    if loan_perc_income > 0.45:
        violations.append("Loan-to-Income ratio exceeds max policy cap of 45%")

    if age < 18:
        violations.append("Applicant is under the minimum age requirement of 18")

    if loan_amt > 50000:
        violations.append("Loan amount exceeds the maximum policy limit of $50,000")

    return {
        "policy_status": "REJECTED" if violations else "APPROVED",
    }

In [ ]:
underwriting_tools = [credit_risk_assessment_tool, check_hard_policy_rules]



In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Initialize LLM with low temperature for consistent, analytical responses
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# System prompt forcing structured financial analysis
system_prompt = """You are a Senior AI Credit Underwriter at a commercial fintech bank.
Your goal is to evaluate loan applications holistically by combining:
1. Machine Learning predictive model outputs (via credit_risk_assessment_tool).
2. Institutional hard policy guardrails (via check_hard_policy_rules).
3. Unstructured qualitative notes submitted by the borrower explaining financial history or temporary income gaps.

Always invoke both tools to gather quantitative facts before forming your decision.

Your final response must strictly use the following format:

### EXECUTIVE CREDIT MEMORANDUM

**Final Recommendation:** [APPROVE / DECLINE / REFER TO HUMAN COMMITTEE]
**Risk Level:** [LOW / MEDIUM / HIGH]
**Model Probability Score:** [X.XX%]

#### 1. Predictive Risk Analysis (XGBoost & SHAP)
- Summarize key quantitative risk drivers derived from the ML model.

#### 2. Policy & Compliance Check
- Summary of hard policy compliance.

#### 3. Qualitative Narrative Evaluation
- Assessment of the applicant's written explanation regarding credit gaps or income anomalies.

#### 4. Final Underwriting Justification
- Concise executive summary detailing why this recommendation was reached.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create the Tool-Calling Agent and Executor
agent = create_tool_calling_agent(llm, underwriting_tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=underwriting_tools, verbose=True)